# Bước 0 — Code, EDA và kiểm tra pipeline
Upload notebook này lên Colab, chọn Runtime → Change runtime type → T4 GPU, rồi Run all.
Notebook tự tạo code, dùng lại dữ liệu đo ngân sách nếu còn (kể cả đường dẫn /kaggle/working/data cũ).
Nó chạy unit tests, EDA, loss ban đầu, overfit một batch, rồi **B01 ResNet-50 10 epoch**, batch 32, seed 0.
Không đánh giá test. Chỉ chạy thí nghiệm thật nếu các kiểm tra pipeline đều đạt.
Sau khi xong, tải **step0_artifacts.zip** trong thanh Files. Checkpoint tải riêng nếu cần giữ lại.


In [ ]:
import sys, subprocess, json, hashlib, importlib.util
from pathlib import Path
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.30', 'fvcore==0.1.5.post20221221', 'matplotlib', 'pandas', 'pillow', 'scikit-learn'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Runtime → Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'eval.py').is_file():
    PROJECT_DIR = PROJECT_DIR.parent
assert (PROJECT_DIR / 'code/train.py').is_file(), 'Chạy notebook trong repo hoặc chỉnh PROJECT_DIR.'
BASE_DIR = Path('/content') if Path('/content').is_dir() else PROJECT_DIR
EDA_DIR = PROJECT_DIR / 'step0'
EPOCHS = 10
BATCH_SIZE = 32
NUM_WORKERS = 2
SEED = 0
RESUME = False


In [ ]:
# Các module top-level này cũng import được trong DataLoader worker (spawn/forkserver).
CODE_DIR = (PROJECT_DIR / 'code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for module_name in ['dataset', 'losses', 'model', 'train', 'step0']:
    sys.modules.pop(module_name, None)
import dataset, losses, model, train, step0


In [ ]:
# Tests chạy CPU trên fixture nhỏ; các con số test không phải kết quả bài lab.
command = [sys.executable, '-X', 'utf8', '-c',
    "import torch,unittest; torch.set_num_threads(2); "
    "s=unittest.defaultTestLoader.discover('tests'); "
    "r=unittest.TextTestRunner(verbosity=2).run(s); raise SystemExit(not r.wasSuccessful())"]
result = subprocess.run(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()


In [ ]:
IMAGES_DIR, LABELS_DIR = step0.prepare_data(BASE_DIR)
eda = step0.analyze_data(IMAGES_DIR, LABELS_DIR, EDA_DIR, seed=SEED)
from IPython.display import display, Image as DisplayImage
for filename in ['class_distribution.png', 'samples_3_per_class.png',
                 'augmentation_examples.png', 'mixup_examples.png', 'cutmix_examples.png']:
    display(DisplayImage(filename=str(EDA_DIR / filename)))


In [ ]:
checks = step0.pipeline_checks(IMAGES_DIR, LABELS_DIR, EDA_DIR,
    backbone='resnet50', seed=SEED)
assert checks['passed'], 'Dừng lại nếu pipeline không qua kiểm tra.'
display(DisplayImage(filename=str(EDA_DIR / 'overfit_curve.png')))
print(json.dumps(checks, ensure_ascii=False, indent=2))


In [ ]:
# B01 là lần train chính thức đầu tiên, không dùng trọng số từ phép overfit.
assert checks['passed'] and eda['reference_filenames_match']
cfg = train.Config(exp_id='B01', backbone='resnet50', seed=SEED,
    epochs=EPOCHS, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR),
    out_dir=str(PROJECT_DIR / 'runs'), pred_dir=str(PROJECT_DIR / 'predictions'),
    curves_dir=str(PROJECT_DIR / 'curves'), save_test_predictions=False, resume=RESUME)
summary = train.run(cfg)
assert not summary['test_evaluated']
prediction = train.ev.read_pred(summary['val_predictions'])
train.ev.check_against_csv(prediction, str(LABELS_DIR / 'val_subset0.csv'), 'val')
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(DisplayImage(filename=summary['curve']))


In [ ]:
step0.write_report(EDA_DIR, eda, checks, summary)
archive = step0.export_artifacts(PROJECT_DIR, BASE_DIR / 'step0_artifacts.zip')
print('Tải file này trong thanh Files:', archive)
print('Checkpoint nếu cần lưu riêng:', train.run_dir(cfg) / 'best.pt',
      'và', train.run_dir(cfg) / 'last.pt')


## Đọc kết quả và bước tiếp theo
ZIP gồm code, unit tests, step0_report.md, bảng đếm lớp, ảnh EDA/augmentation/overfit,
history.csv, config.json, val_logits.npy, file dự đoán val và đường cong B01.
Đọc ảnh mẫu rồi bổ sung nhận xét trong báo cáo: Chinee Apple/Snake Weed, Negatives, và crop/CutMix.
Chưa có kết luận backbone nào tốt nhất; Bước 1 chạy 4 backbone còn lại với **cùng** công thức, seed và số epoch.
Không chạy notebook đo ngân sách lần nữa. Không bật save_test_predictions ở Bước 0–3.
Nếu phiên bị ngắt mà last.pt còn tồn tại, giữ nguyên cấu hình, đặt RESUME=True và chạy lại ô baseline.
Nếu đổi batch/epoch cho B01, các backbone sau phải dùng cùng cấu hình để so sánh công bằng.
